In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

TRAIN_SZ = 0.8
VALID_SZ = 1.0 - TRAIN_SZ

# Read the data
train = pd.read_csv('../datasets/home-data-for-ml-course/train.csv', index_col='Id')
test = pd.read_csv('../datasets/home-data-for-ml-course/test.csv', index_col='Id')

# Obtain target and predictors
features = ['LotArea', 'YearBuilt', '1stFlrSF', '2ndFlrSF', 'FullBath', 'BedroomAbvGr', 'TotRmsAbvGrd']
X = train[features].copy()
y = train.SalePrice.copy()

X_test = test[features].copy()

# Break off validation set from training data
X_train, X_valid, y_train, y_valid = train_test_split(X, y, 
                                                      train_size=TRAIN_SZ, test_size=VALID_SZ,
                                                      random_state=0)

In [2]:
X_train.head()

,LotArea,YearBuilt,1stFlrSF,2ndFlrSF,FullBath,BedroomAbvGr,TotRmsAbvGrd
Id,,,,,,,
619,11694,2007,1828,0,2,3,9
871,6600,1962,894,0,1,2,5
93,13360,1921,964,0,1,2,5
818,13265,2002,1689,0,2,3,7
303,13704,2001,1541,0,2,3,6


In [3]:
X_valid.head()

,LotArea,YearBuilt,1stFlrSF,2ndFlrSF,FullBath,BedroomAbvGr,TotRmsAbvGrd
Id,,,,,,,
530,32668,1957,2515,0,3,4,9
492,9490,1941,958,620,1,3,5
460,7015,1950,979,224,1,3,5
280,10005,1977,1156,866,2,4,8
656,1680,1971,525,567,1,3,6


In [4]:
X_test.head()

,LotArea,YearBuilt,1stFlrSF,2ndFlrSF,FullBath,BedroomAbvGr,TotRmsAbvGrd
Id,,,,,,,
1461,11622,1961,896,0,1,2,5
1462,14267,1958,1329,0,1,3,6
1463,13830,1997,928,701,2,3,6
1464,9978,1998,926,678,2,3,7
1465,5005,1992,1280,0,2,2,5


In [5]:
from sklearn.ensemble import RandomForestRegressor

# Define the models
model_1 = RandomForestRegressor(n_estimators=50, random_state=0)
model_2 = RandomForestRegressor(n_estimators=100, random_state=0)
model_3 = RandomForestRegressor(n_estimators=100, criterion='absolute_error', random_state=0)
model_4 = RandomForestRegressor(n_estimators=200, min_samples_split=20, random_state=0)
model_5 = RandomForestRegressor(n_estimators=100, max_depth=7, random_state=0)

models = [model_1, model_2, model_3, model_4, model_5]

In [6]:
import math
from sklearn.metrics import mean_absolute_error

# Function for comparing different models
def score_model(model, X_t=X_train, X_v=X_valid, y_t=y_train, y_v=y_valid):
    model.fit(X_t, y_t)
    preds = model.predict(X_v)
    return mean_absolute_error(y_v, preds)

# Pick best model
model_no, min_mae = 0, math.inf
for i, model in enumerate(models, start=1):
    mae = score_model(model)
    print(f"Model {i} \t\t MAE: {mae}")
    if mae < min_mae:
        min_mae = mae
        model_no = i

print(f"\nBest model: {model_no} \t\t MAE: {min_mae}")


Model 1 		 MAE: 24015.492818003917
Model 2 		 MAE: 23740.979228636657
Model 3 		 MAE: 23431.747157534246
Model 4 		 MAE: 23996.676789668687
Model 5 		 MAE: 23706.672864217904

Best model: 3 		 MAE: 23431.747157534246


In [7]:
# Define and fit the best model
best_model = model[model_no]
best_model.fit(X, y)

# Generate test predictions
y_test = best_model.predict(X_test)

# Save predictions in format used for competition scoring
output = pd.DataFrame({'Id': X_test.index, 'SalePrice': y_test})
# output.to_csv('submission.csv', index=False)

In [9]:
from IPython.display import display, HTML

results = pd.DataFrame(y_test, columns=['SalePrice'], dtype='float64')

report = f"""
<div style="display: flex;">
    <div style="border-right: #fcfcfc 1px solid;">
        {X_test.to_html(index=False)}
    </div>
    <div style="background-color: #f0f6fe;">
        {results.to_html(index=False)}
    </div>
</div>
"""

display(HTML(report))


LotArea,YearBuilt,1stFlrSF,2ndFlrSF,FullBath,BedroomAbvGr,TotRmsAbvGrd
11622,1961,896,0,1,2,5
14267,1958,1329,0,1,3,6
13830,1997,928,701,2,3,6
9978,1998,926,678,2,3,7
5005,1992,1280,0,2,2,5
10000,1993,763,892,2,3,7
7980,1992,1187,0,2,3,6
8402,1998,789,676,2,3,7
10176,1990,1341,0,1,2,5
8400,1970,882,0,1,2,4
